In [1]:
import os
import pandas as pd
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer

In [2]:
import time
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             confusion_matrix)
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import SVC, LinearSVC 
from sklearn.ensemble import AdaBoostClassifier, GradientBoostingClassifier, RandomForestClassifier
from tensorflow.keras.layers import Conv1D, MaxPooling1D, LSTM, Dropout, Dense
from tabulate import tabulate
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.sequence import pad_sequences
import tensorflow as tf
import glob
from sklearn.tree import DecisionTreeClassifier

In [3]:
rows=7000

In [4]:
mirai_df_list = []
for file in glob.glob("./Danmini/mirai/*.csv"):
    tmp_df = pd.read_csv(file, nrows=rows)
    tmp_df["target"] = "mirai-"+ os.path.splitext(os.path.basename(file))[0]
    mirai_df_list.append(tmp_df)

In [5]:
mirai_df_list[0].head()

,MI_dir_L5_weight,MI_dir_L5_mean,MI_dir_L5_variance,MI_dir_L3_weight,MI_dir_L3_mean,MI_dir_L3_variance,MI_dir_L1_weight,MI_dir_L1_mean,MI_dir_L1_variance,MI_dir_L0.1_weight,...,HpHp_L0.1_covariance,HpHp_L0.1_pcc,HpHp_L0.01_weight,HpHp_L0.01_mean,HpHp_L0.01_std,HpHp_L0.01_magnitude,HpHp_L0.01_radius,HpHp_L0.01_covariance,HpHp_L0.01_pcc,target
0,1.000000,566.0,0.000000e+00,1.000000,566.0,0.000000e+00,1.000000,566.0,0.000000e+00,1.000000,...,0.0,0.0,1.0,566.0,0.0,566.0,0.0,0.0,0.0,mirai-ack
1,1.996585,566.0,5.820766e-11,1.997950,566.0,5.820766e-11,1.999316,566.0,0.000000e+00,1.999932,...,0.0,0.0,1.0,566.0,0.0,566.0,0.0,0.0,0.0,mirai-ack
2,2.958989,566.0,0.000000e+00,2.975291,566.0,5.820766e-11,2.991729,566.0,5.820766e-11,2.999171,...,0.0,0.0,1.0,566.0,0.0,566.0,0.0,0.0,0.0,mirai-ack
3,3.958979,566.0,0.000000e+00,3.975285,566.0,0.000000e+00,3.991727,566.0,1.164153e-10,3.999171,...,0.0,0.0,1.0,566.0,0.0,566.0,0.0,0.0,0.0,mirai-ack
4,4.914189,566.0,1.164153e-10,4.948239,566.0,5.820766e-11,4.982654,566.0,5.820766e-11,4.998261,...,0.0,0.0,1.0,566.0,0.0,566.0,0.0,0.0,0.0,mirai-ack


In [6]:
mirai_df_list[0].info()
mirai_df_list[0].columns

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7000 entries, 0 to 6999
Columns: 116 entries, MI_dir_L5_weight to target
dtypes: float64(115), object(1)
memory usage: 6.2+ MB


Index(['MI_dir_L5_weight', 'MI_dir_L5_mean', 'MI_dir_L5_variance',
       'MI_dir_L3_weight', 'MI_dir_L3_mean', 'MI_dir_L3_variance',
       'MI_dir_L1_weight', 'MI_dir_L1_mean', 'MI_dir_L1_variance',
       'MI_dir_L0.1_weight',
       ...
       'HpHp_L0.1_covariance', 'HpHp_L0.1_pcc', 'HpHp_L0.01_weight',
       'HpHp_L0.01_mean', 'HpHp_L0.01_std', 'HpHp_L0.01_magnitude',
       'HpHp_L0.01_radius', 'HpHp_L0.01_covariance', 'HpHp_L0.01_pcc',
       'target'],
      dtype='object', length=116)

In [7]:
gafgyt_df_list = []
for file in glob.glob("./Danmini/gafgyt/*.csv"):
    tmp_df = pd.read_csv(file, nrows=rows)
    tmp_df["target"] = "gafgyt-"+ os.path.splitext(os.path.basename(file))[0]
    gafgyt_df_list.append(tmp_df)
gafgyt_df_list[0].head()

benign_df = pd.read_csv("./Danmini/benign_traffic.csv", nrows=rows)
benign_df["target"] = "Benign"

gafgyt_df_list.append(benign_df)
df_list = mirai_df_list + gafgyt_df_list
df = pd.concat(df_list)
df.head()

,MI_dir_L5_weight,MI_dir_L5_mean,MI_dir_L5_variance,MI_dir_L3_weight,MI_dir_L3_mean,MI_dir_L3_variance,MI_dir_L1_weight,MI_dir_L1_mean,MI_dir_L1_variance,MI_dir_L0.1_weight,...,HpHp_L0.1_covariance,HpHp_L0.1_pcc,HpHp_L0.01_weight,HpHp_L0.01_mean,HpHp_L0.01_std,HpHp_L0.01_magnitude,HpHp_L0.01_radius,HpHp_L0.01_covariance,HpHp_L0.01_pcc,target
0,1.000000,566.0,0.000000e+00,1.000000,566.0,0.000000e+00,1.000000,566.0,0.000000e+00,1.000000,...,0.0,0.0,1.0,566.0,0.0,566.0,0.0,0.0,0.0,mirai-ack
1,1.996585,566.0,5.820766e-11,1.997950,566.0,5.820766e-11,1.999316,566.0,0.000000e+00,1.999932,...,0.0,0.0,1.0,566.0,0.0,566.0,0.0,0.0,0.0,mirai-ack
2,2.958989,566.0,0.000000e+00,2.975291,566.0,5.820766e-11,2.991729,566.0,5.820766e-11,2.999171,...,0.0,0.0,1.0,566.0,0.0,566.0,0.0,0.0,0.0,mirai-ack
3,3.958979,566.0,0.000000e+00,3.975285,566.0,0.000000e+00,3.991727,566.0,1.164153e-10,3.999171,...,0.0,0.0,1.0,566.0,0.0,566.0,0.0,0.0,0.0,mirai-ack
4,4.914189,566.0,1.164153e-10,4.948239,566.0,5.820766e-11,4.982654,566.0,5.820766e-11,4.998261,...,0.0,0.0,1.0,566.0,0.0,566.0,0.0,0.0,0.0,mirai-ack


In [8]:
X = df.drop(columns=['target'])  
y = df['target']  

In [9]:
# folder_path = './mirai'

# files = [file for file in os.listdir(folder_path) if file.endswith('.csv')]

# datasets = [pd.read_csv(os.path.join(folder_path, file)) for file in files]

# print(f"Loaded {len(datasets)} datasets")
# print("\nSample from the first dataset:")
# datasets[0].head()


In [10]:
# def preprocess_data(df):
#     df = df.drop(['src_ip', 'dst_ip', 'dns_query'], axis=1, errors='ignore')
    
#     df.fillna(0, inplace=True)
    
#     label_encoders = {}
#     for column in df.select_dtypes(include=['object']).columns:
#         le = LabelEncoder()
#         df[column] = le.fit_transform(df[column])
#         label_encoders[column] = le
    
#     if 'target' not in df.columns:
#         # Set a default target if it's missing. Replace with actual logic if needed.
#         df['target'] = 0  # Assuming 0 as default, modify this according to your case.
    
#     return df

In [11]:

# # Preprocess all datasets
# cleaned_datasets = [preprocess_data(dataset) for dataset in datasets]

# print("\nSample from the cleaned first dataset:")
# print(cleaned_datasets[0].head())

# # Concatenate all cleaned datasets
# combined_data = pd.concat(cleaned_datasets, axis=0)

# # Separate features and target
# X = combined_data.drop('target', axis=1)  # Ensure the target column is present before dropping
# y = combined_data['target']

# print("\nCombined Data:")
# print(combined_data.head())

In [12]:
# def preprocess_data(df):
#     df = df.drop(['src_ip', 'dst_ip', 'dns_query'], axis=1, errors='ignore')
    
#     df.fillna(0, inplace=True)
    
#     label_encoders = {}
#     for column in df.select_dtypes(include=['object']).columns:
#         le = LabelEncoder()
#         df[column] = le.fit_transform(df[column])
#         label_encoders[column] = le

#     return df

# cleaned_datasets = [preprocess_data(dataset) for dataset in datasets]

# print("\nSample from the cleaned first dataset:")
# cleaned_datasets[0].head()

In [13]:

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)


In [14]:
# print(combined_data.columns)

In [15]:
# combined_data.head()

In [16]:
def calculate_metrics(y_test, y_pred, training_time, average='weighted', model_name='Random Forest'):
    accuracy = accuracy_score(y_test, y_pred)
    error_rate = 1 - accuracy
    precision = precision_score(y_test, y_pred, average=average)
    recall = recall_score(y_test, y_pred, average=average)
    f1 = f1_score(y_test, y_pred, average=average)

    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel() if len(confusion_matrix(y_test, y_pred).ravel()) == 4 else (None, None, None, None)
    
    false_positive_rate = fp / (fp + tn) if fp is not None else "N/A"
    specificity = tn / (tn + fp) if tn is not None else "N/A"

    headers = ["Model", "Training Time (s)", "Accuracy (%)", "Error Rate (%)", 
               "Precision", "Recall", "F1-Score", "False Positive Rate", "Specificity"]
    data = [[
        model_name, 
        f"{training_time:.2f}", 
        f"{accuracy * 100:.2f}", 
        f"{error_rate * 100:.2f}", 
        f"{precision:.2f}", 
        f"{recall:.2f}", 
        f"{f1:.2f}", 
        f"{false_positive_rate:.2f}" if false_positive_rate != "N/A" else "0", 
        f"{specificity:.2f}" if specificity != "N/A" else "0"
    ]]

    return tabulate(data, headers=headers, tablefmt="grid")


In [17]:
def train_model(X_train, y_train, X_test):
    start_time = time.time()  

    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)

    training_time = time.time() - start_time

    y_pred = model.predict(X_test)

    return y_pred, training_time

y_pred, training_time = train_model(X_train, y_train, X_test)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Random Forest')
print(metrics_table)

+---------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model         |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+===============+=====================+================+==================+=============+==========+============+=======================+===============+
| Random Forest |                15.6 |          99.81 |             0.19 |           1 |        1 |          1 |                     0 |             0 |
+---------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [18]:
def train_logistic_regression(X_train, y_train, X_test):
    start_time = time.time()

    model = LogisticRegression(random_state=42, max_iter=1000)
    model.fit(X_train, y_train)

    training_time = time.time() - start_time

    y_pred = model.predict(X_test)

    return y_pred, training_time
y_pred, training_time = train_logistic_regression(X_train, y_train, X_test)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Logistic Regression')
print(metrics_table)

c:\Users\pc\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\linear_model\_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
c:\Users\pc\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


+---------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model               |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+=====================+=====================+================+==================+=============+==========+============+=======================+===============+
| Logistic Regression |               29.17 |          17.79 |            82.21 |        0.18 |     0.18 |        0.1 |                     0 |             0 |
+---------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [19]:

def train_svm(X_train, y_train, X_test):
    start_time = time.time()

    model = LinearSVC(random_state=42, max_iter=1000, C=0.5, dual=False)  # dual=False speeds up linear SVM
    model.fit(X_train, y_train)

    training_time = time.time() - start_time

    y_pred = model.predict(X_test)

    return y_pred, training_time

y_pred, training_time = train_svm(X_train, y_train, X_test)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Support Vector Machine')
print(metrics_table)

c:\Users\pc\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


+------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model                  |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+========================+=====================+================+==================+=============+==========+============+=======================+===============+
| Support Vector Machine |               25.91 |           9.31 |            90.69 |        0.12 |     0.09 |       0.02 |                     0 |             0 |
+------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [20]:
def train_sgd(X_train, y_train, X_test):
    start_time = time.time()

    model = SGDClassifier(max_iter=1000, tol=1e-3, random_state=42)
    model.fit(X_train, y_train)

    training_time = time.time() - start_time

    y_pred = model.predict(X_test)

    return y_pred, training_time

y_pred, training_time = train_sgd(X_train, y_train, X_test)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Stochastic Gradient Descent')
print(metrics_table)

c:\Users\pc\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


+-----------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model                       |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+=============================+=====================+================+==================+=============+==========+============+=======================+===============+
| Stochastic Gradient Descent |                4.79 |          20.49 |            79.51 |        0.17 |      0.2 |       0.12 |                     0 |             0 |
+-----------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [21]:
def train_adaboost(X_train, y_train, X_test):
    start_time = time.time()

    model = AdaBoostClassifier(random_state=42)
    model.fit(X_train, y_train)
    
    training_time = time.time() - start_time

    y_pred = model.predict(X_test)

    return y_pred, training_time

y_pred, training_time = train_adaboost(X_train, y_train, X_test)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Ada Boost')
print(metrics_table)

c:\Users\pc\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
c:\Users\pc\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


+-----------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model     |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+===========+=====================+================+==================+=============+==========+============+=======================+===============+
| Ada Boost |               26.64 |          35.68 |            64.32 |        0.33 |     0.36 |       0.27 |                     0 |             0 |
+-----------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [22]:
def train_gradient_boosting(X_train, y_train, X_test):
    start_time = time.time()
    _tune=0.004
    model = GradientBoostingClassifier(random_state=42)
    model.fit(X_train, y_train)

    training_time = time.time() - start_time

    y_pred = model.predict(X_test)

    return y_pred, training_time
y_pred, training_time = train_gradient_boosting(X_train, y_train, X_test)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Gradient Boosting')
print(metrics_table)

+-------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model             |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+===================+=====================+================+==================+=============+==========+============+=======================+===============+
| Gradient Boosting |               17.85 |          99.81 |             0.19 |           1 |        1 |          1 |                     0 |             0 |
+-------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [23]:
y_train_str = y_train.astype(str)
y_test_str = y_test.astype(str)

combined_labels = np.concatenate((y_train_str, y_test_str))

label_encoder = LabelEncoder()
label_encoder.fit(combined_labels)

y_train_encoded = label_encoder.transform(y_train_str)
y_test_encoded = label_encoder.transform(y_test_str)

    # label_encoder = LabelEncoder()
    # y_train_encoded = label_encoder.fit_transform(y_train.astype(str))  
    # y_test_encoded = label_encoder.transform(y_test.astype(str))  

In [24]:
from keras.callbacks import EarlyStopping

def train_ann(X_train, y_train, X_test, y_test, input_dim, num_classes):
    start_time = time.time()

    model = Sequential()
    model.add(Dense(32, input_dim=input_dim, activation='relu'))  # Reduced size
    model.add(Dense(32, activation='relu'))  
    model.add(Dense(num_classes, activation='softmax')) 

    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

    model.fit(X_train, y_train, epochs=15, batch_size=64, verbose=0, validation_data=(X_test, y_test), callbacks=[early_stopping])  # Fewer epochs and larger batch size

    training_time = time.time() - start_time

    y_pred_prob = model.predict(X_test)
    y_pred = np.argmax(y_pred_prob, axis=1) 
    accuracy = accuracy_score(y_test, y_pred)
    print(f'Accuracy: {accuracy * 100:.2f}%')

    return y_pred, training_time

num_classes = len(set(y_train.astype(str)))  
y_pred, training_time = train_ann(X_train, y_train_encoded, X_test, y_test_encoded, input_dim=X_train.shape[1], num_classes=num_classes)
metrics_table = calculate_metrics(y_test_encoded, y_pred, training_time, 'weighted', 'Artificial Neural Network')
print(metrics_table)


c:\Users\pc\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


722/722 ━━━━━━━━━━━━━━━━━━━━ 1s 676us/step
Accuracy: 19.27%
+---------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model                     |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+===========================+=====================+================+==================+=============+==========+============+=======================+===============+
| Artificial Neural Network |                15.6 |          19.27 |            80.73 |        0.19 |     0.19 |       0.14 |                     0 |             0 |
+---------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


c:\Users\pc\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [25]:
def train_deep_ann(X_train, y_train, X_test, y_test, input_dim, num_classes):
    start_time = time.time()

    model = Sequential()
    model.add(Dense(64, input_dim=input_dim, activation='relu')) 
    model.add(Dropout(0.2))
    model.add(Dense(64, activation='relu'))
    model.add(Dense(32, activation='relu')) 
    model.add(Dense(num_classes, activation='softmax'))

    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

    model.fit(X_train, y_train, epochs=20, batch_size=64, verbose=0, validation_data=(X_test, y_test), callbacks=[early_stopping])  

    training_time = time.time() - start_time

    y_pred_prob = model.predict(X_test)
    y_pred = np.argmax(y_pred_prob, axis=1)

    accuracy = accuracy_score(y_test, y_pred)
    print(f'Accuracy: {accuracy * 100:.2f}%')

    return y_pred, training_time

num_classes = len(set(y_train.astype(str)))
y_pred, training_time = train_deep_ann(X_train, y_train_encoded, X_test, y_test_encoded, input_dim=X_train.shape[1], num_classes=num_classes)
metrics_table = calculate_metrics(y_test_encoded, y_pred, training_time, 'weighted', 'Deep Artificial Neural Network')
print(metrics_table)

c:\Users\pc\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


722/722 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step
Accuracy: 26.36%
+--------------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model                          |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+================================+=====================+================+==================+=============+==========+============+=======================+===============+
| Deep Artificial Neural Network |               28.58 |          26.36 |            73.64 |        0.13 |     0.26 |       0.15 |                     0 |             0 |
+--------------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


c:\Users\pc\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [26]:
def train_cnn_lstm(X_train, y_train, X_test, y_test, input_shape, num_classes):
    start_time = time.time()

    model = Sequential()

    model.add(Conv1D(filters=64, kernel_size=3, activation='relu', input_shape=input_shape))
    model.add(MaxPooling1D(pool_size=2))
    model.add(Dropout(0.3))

    model.add(Conv1D(filters=64, kernel_size=3, activation='relu'))
    model.add(MaxPooling1D(pool_size=2))
    model.add(Dropout(0.3))

    model.add(LSTM(50, activation='relu', return_sequences=False))
    model.add(Dropout(0.3))

    model.add(Dense(50, activation='relu'))
    model.add(Dense(num_classes, activation='softmax'))

    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

    model.fit(X_train, y_train, epochs=20, batch_size=64, verbose=1, validation_split=0.2, callbacks=[early_stopping])

    training_time = time.time() - start_time

    y_pred_prob = model.predict(X_test)
    y_pred = np.argmax(y_pred_prob, axis=1)

    accuracy = accuracy_score(y_test, y_pred)
    print(f'Accuracy: {accuracy * 100:.2f}%')

    return y_pred, training_time

label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

X_train_cnn_lstm = X_train.to_numpy().reshape((X_train.shape[0], X_train.shape[1], 1))
X_test_cnn_lstm = X_test.to_numpy().reshape((X_test.shape[0], X_test.shape[1], 1))

num_classes = len(set(y_train_encoded))

y_pred, training_time = train_cnn_lstm(X_train_cnn_lstm, y_train_encoded, X_test_cnn_lstm, y_test_encoded, input_shape=(X_train.shape[1], 1), num_classes=num_classes)

metrics_table = calculate_metrics(y_test_encoded, y_pred, training_time, 'weighted', 'Convolutional Neural Network')
print(metrics_table)


c:\Users\pc\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/20
674/674 ━━━━━━━━━━━━━━━━━━━━ 12s 15ms/step - accuracy: 0.0900 - loss: 119804589506560.0000 - val_accuracy: 0.0902 - val_loss: 170268336.0000
Epoch 2/20
674/674 ━━━━━━━━━━━━━━━━━━━━ 9s 13ms/step - accuracy: 0.1325 - loss: 448120356864.0000 - val_accuracy: 0.0883 - val_loss: 157482.6406
Epoch 3/20
674/674 ━━━━━━━━━━━━━━━━━━━━ 9s 13ms/step - accuracy: 0.1400 - loss: 27581456384.0000 - val_accuracy: 0.1604 - val_loss: 292.4600
Epoch 4/20
674/674 ━━━━━━━━━━━━━━━━━━━━ 10s 15ms/step - accuracy: 0.1391 - loss: 17672941568.0000 - val_accuracy: 0.1437 - val_loss: 398699.8125
Epoch 5/20
674/674 ━━━━━━━━━━━━━━━━━━━━ 9s 13ms/step - accuracy: 0.1355 - loss: 15577108480.0000 - val_accuracy: 0.1660 - val_loss: 4863.2734
Epoch 6/20
674/674 ━━━━━━━━━━━━━━━━━━━━ 9s 13ms/step - accuracy: 0.1315 - loss: 12725910528.0000 - val_accuracy: 0.1724 - val_loss: 107.5821
Epoch 7/20
674/674 ━━━━━━━━━━━━━━━━━━━━ 9s 14ms/step - accuracy: 0.1156 - loss: 3861021440.0000 - val_accuracy: 0.1517 - val_loss: 14.

c:\Users\pc\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


## NORMALIZATION OF X INPUT

In [27]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [28]:
def train_model(X_train, y_train, X_test):
    start_time = time.time()  

    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)

    training_time = time.time() - start_time

    y_pred = model.predict(X_test)

    return y_pred, training_time

y_pred, training_time = train_model(X_train, y_train, X_test)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Random Forest')
print(metrics_table)

+---------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model         |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+===============+=====================+================+==================+=============+==========+============+=======================+===============+
| Random Forest |               12.88 |          91.04 |             8.96 |        0.91 |     0.91 |       0.88 |                     0 |             0 |
+---------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [29]:
def train_logistic_regression(X_train, y_train, X_test):
    start_time = time.time()

    model = LogisticRegression(random_state=42, max_iter=1000)
    model.fit(X_train, y_train)

    training_time = time.time() - start_time

    y_pred = model.predict(X_test)

    return y_pred, training_time
y_pred, training_time = train_logistic_regression(X_train, y_train, X_test)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Logistic Regression')
print(metrics_table)

+---------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model               |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+=====================+=====================+================+==================+=============+==========+============+=======================+===============+
| Logistic Regression |               14.81 |          87.28 |            12.72 |        0.83 |     0.87 |       0.84 |                     0 |             0 |
+---------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [30]:
def train_svm(X_train, y_train, X_test):
    start_time = time.time()

    model = SVC(random_state=42)
    model.fit(X_train, y_train)

    training_time = time.time() - start_time

    y_pred = model.predict(X_test)

    return y_pred, training_time

y_pred, training_time = train_svm(X_train, y_train, X_test)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Support Vector Machine')
print(metrics_table)

c:\Users\pc\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


+------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model                  |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+========================+=====================+================+==================+=============+==========+============+=======================+===============+
| Support Vector Machine |                33.8 |           89.4 |             10.6 |        0.85 |     0.89 |       0.86 |                     0 |             0 |
+------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [31]:
def train_sgd(X_train, y_train, X_test):
    start_time = time.time()

    model = SGDClassifier(random_state=42)
    model.fit(X_train, y_train)

    training_time = time.time() - start_time

    y_pred = model.predict(X_test)

    return y_pred, training_time
y_pred, training_time = train_sgd(X_train, y_train, X_test)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Stochastic Gradient Descent')
print(metrics_table)

c:\Users\pc\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


+-----------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model                       |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+=============================+=====================+================+==================+=============+==========+============+=======================+===============+
| Stochastic Gradient Descent |                 1.9 |          84.82 |            15.18 |        0.81 |     0.85 |       0.82 |                     0 |             0 |
+-----------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [32]:
def train_adaboost(X_train, y_train, X_test):
    start_time = time.time()

    model = AdaBoostClassifier(random_state=42)
    model.fit(X_train, y_train)

    training_time = time.time() - start_time

    y_pred = model.predict(X_test)

    return y_pred, training_time

y_pred, training_time = train_adaboost(X_train, y_train, X_test)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Ada Boost')
print(metrics_table)

c:\Users\pc\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\ensemble\_weight_boosting.py:527: FutureWarning: The SAMME.R algorithm (the default) is deprecated and will be removed in 1.6. Use the SAMME algorithm to circumvent this warning.
  warnings.warn(
c:\Users\pc\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


+-----------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model     |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+===========+=====================+================+==================+=============+==========+============+=======================+===============+
| Ada Boost |               21.73 |          35.68 |            64.32 |        0.33 |     0.36 |       0.27 |                     0 |             0 |
+-----------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [47]:
def train_gradient_boosting(X_train, y_train, X_test):
    start_time = time.time()

    model = GradientBoostingClassifier(random_state=42)
    model.fit(X_train, y_train)

    training_time = time.time() - start_time

    y_pred = model.predict(X_test)

    return y_pred, training_time
y_pred, training_time = train_gradient_boosting(X_train, y_train, X_test)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Gradient Boosting')
print(metrics_table)

In [51]:
def train_ann(X_train, y_train, X_test, y_test, input_dim, num_classes):
    start_time = time.time()

    model = Sequential()
    model.add(Dense(64, input_dim=input_dim, activation='relu'))  
    model.add(Dense(64, activation='relu'))  
    model.add(Dense(num_classes, activation='softmax')) 

    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    model.fit(X_train, y_train, epochs=10, batch_size=32, verbose=0)

    training_time = time.time() - start_time

    y_pred_prob = model.predict(X_test)
    y_pred = np.argmax(y_pred_prob, axis=1) 
    accuracy = accuracy_score(y_test, y_pred)
    print(f'Accuracy: {accuracy * 100:.2f}%')

    return y_pred, training_time

num_classes = len(set(y_train.astype(str)))  
y_pred, training_time = train_ann(X_train, y_train_encoded, X_test, y_test_encoded, input_dim=X_train.shape[1], num_classes=num_classes)
metrics_table = calculate_metrics(y_test_encoded, y_pred, training_time, 'weighted', 'Artificial Neural Network')
print(metrics_table)

c:\Users\pc\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


9547/9547 ━━━━━━━━━━━━━━━━━━━━ 7s 720us/step
Accuracy: 90.51%


In [54]:
def train_deep_ann(X_train, y_train, X_test, y_test, input_dim, num_classes):
    start_time = time.time()


    model = Sequential()
    model.add(Dense(128, input_dim=input_dim, activation='relu')) 
    model.add(Dropout(0.2)) 
    model.add(Dense(128, activation='relu')) 
    model.add(Dense(64, activation='relu')) 
    model.add(Dense(32, activation='relu'))
    model.add(Dense(num_classes, activation='softmax'))  

    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    model.fit(X_train, y_train, epochs=20, batch_size=32, verbose=0)

    training_time = time.time() - start_time

    y_pred_prob = model.predict(X_test)
    y_pred = np.argmax(y_pred_prob, axis=1)

    accuracy = accuracy_score(y_test, y_pred)
    print(f'Accuracy: {accuracy * 100:.2f}%')

    return y_pred, training_time

num_classes = len(set(y_train.astype(str)))
y_pred, training_time = train_deep_ann(X_train, y_train_encoded, X_test, y_test_encoded, input_dim=X_train.shape[1], num_classes=num_classes)
metrics_table = calculate_metrics(y_test_encoded, y_pred, training_time, 'weighted', 'Deep Artificial Neural Network')
print(metrics_table)

c:\Users\pc\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


9547/9547 ━━━━━━━━━━━━━━━━━━━━ 8s 795us/step
Accuracy: 79.81%


In [33]:
def train_cnn_lstm(X_train, y_train, X_test, y_test, input_shape, num_classes):
    start_time = time.time()

    model = Sequential()

    model.add(Conv1D(filters=64, kernel_size=3, activation='relu', input_shape=input_shape))
    model.add(MaxPooling1D(pool_size=2))
    model.add(Dropout(0.3))

    model.add(Conv1D(filters=64, kernel_size=3, activation='relu'))
    model.add(MaxPooling1D(pool_size=2))
    model.add(Dropout(0.3))

    model.add(LSTM(50, activation='relu', return_sequences=False))
    model.add(Dropout(0.3))

    model.add(Dense(50, activation='relu'))
    model.add(Dense(num_classes, activation='softmax'))

    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

    model.fit(X_train, y_train, epochs=20, batch_size=64, verbose=1, validation_split=0.2, callbacks=[early_stopping])

    training_time = time.time() - start_time

    y_pred_prob = model.predict(X_test)
    y_pred = np.argmax(y_pred_prob, axis=1)

    accuracy = accuracy_score(y_test, y_pred)
    print(f'Accuracy: {accuracy * 100:.2f}%')

    return y_pred, training_time

label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

X_train_cnn_lstm = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))
X_test_cnn_lstm = X_test.reshape((X_test.shape[0], X_test.shape[1], 1))

num_classes = len(set(y_train_encoded))

y_pred, training_time = train_cnn_lstm(X_train_cnn_lstm, y_train_encoded, X_test_cnn_lstm, y_test_encoded, input_shape=(X_train.shape[1], 1), num_classes=num_classes)

metrics_table = calculate_metrics(y_test_encoded, y_pred, training_time, 'weighted', 'Convolutional Neural Network')
print(metrics_table)


Epoch 1/20


c:\Users\pc\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


674/674 ━━━━━━━━━━━━━━━━━━━━ 12s 14ms/step - accuracy: 0.3611 - loss: 1.5180 - val_accuracy: 0.6343 - val_loss: 0.7559
Epoch 2/20
674/674 ━━━━━━━━━━━━━━━━━━━━ 10s 15ms/step - accuracy: 0.7081 - loss: 0.5301 - val_accuracy: 0.7268 - val_loss: 0.4790
Epoch 3/20
674/674 ━━━━━━━━━━━━━━━━━━━━ 9s 14ms/step - accuracy: 0.6948 - loss: 0.5814 - val_accuracy: 0.7291 - val_loss: 0.4552
Epoch 4/20
674/674 ━━━━━━━━━━━━━━━━━━━━ 10s 14ms/step - accuracy: 0.7457 - loss: 0.4451 - val_accuracy: 0.7329 - val_loss: 0.4348
Epoch 5/20
674/674 ━━━━━━━━━━━━━━━━━━━━ 9s 14ms/step - accuracy: 0.7587 - loss: 0.4351 - val_accuracy: 0.7697 - val_loss: 0.3762
Epoch 6/20
674/674 ━━━━━━━━━━━━━━━━━━━━ 10s 15ms/step - accuracy: 0.7828 - loss: 0.3818 - val_accuracy: 0.7609 - val_loss: 0.3831
Epoch 7/20
674/674 ━━━━━━━━━━━━━━━━━━━━ 10s 14ms/step - accuracy: 0.7846 - loss: 0.3769 - val_accuracy: 0.8109 - val_loss: 0.3401
Epoch 8/20
674/674 ━━━━━━━━━━━━━━━━━━━━ 10s 15ms/step - accuracy: 0.7892 - loss: 0.3705 - val_accuracy:

c:\Users\pc\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


## USING BEST FEATURES

In [34]:
from sklearn.feature_selection import f_classif, SelectKBest

fs = SelectKBest(f_classif, k=28)
fs.fit(X_train, y_train)
x_train = fs.transform(X_train)
x_test = fs.transform(X_test)

In [16]:
def train_model(X_train, y_train, X_test):
    start_time = time.time()  

    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)

    training_time = time.time() - start_time

    y_pred = model.predict(X_test)

    return y_pred, training_time

y_pred, training_time = train_model(X_train, y_train, X_test)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Random Forest')
print(metrics_table)

+---------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model         |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+===============+=====================+================+==================+=============+==========+============+=======================+===============+
| Random Forest |              207.74 |          90.93 |             9.07 |        0.94 |     0.91 |       0.88 |                     0 |             0 |
+---------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [17]:
def train_logistic_regression(X_train, y_train, X_test):
    start_time = time.time()

    model = LogisticRegression(random_state=42, max_iter=1000)
    model.fit(X_train, y_train)

    training_time = time.time() - start_time

    y_pred = model.predict(X_test)

    return y_pred, training_time
y_pred, training_time = train_logistic_regression(X_train, y_train, X_test)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Logistic Regression')
print(metrics_table)

+---------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model               |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+=====================+=====================+================+==================+=============+==========+============+=======================+===============+
| Logistic Regression |              211.52 |          90.93 |             9.07 |        0.94 |     0.91 |       0.88 |                     0 |             0 |
+---------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [18]:
def train_svm(X_train, y_train, X_test):
    start_time = time.time()

    model = SVC(random_state=42)
    model.fit(X_train, y_train)

    training_time = time.time() - start_time

    y_pred = model.predict(X_test)

    return y_pred, training_time

y_pred, training_time = train_svm(X_train, y_train, X_test)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Support Vector Machine')
print(metrics_table)

+------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model                  |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+========================+=====================+================+==================+=============+==========+============+=======================+===============+
| Support Vector Machine |              204.56 |          90.93 |             9.07 |        0.94 |     0.91 |       0.88 |                     0 |             0 |
+------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [19]:
def train_sgd(X_train, y_train, X_test):
    start_time = time.time()

    model = SGDClassifier(random_state=42)
    model.fit(X_train, y_train)

    training_time = time.time() - start_time

    y_pred = model.predict(X_test)

    return y_pred, training_time
y_pred, training_time = train_sgd(X_train, y_train, X_test)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Stochastic Gradient Descent')
print(metrics_table)

+-----------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model                       |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+=============================+=====================+================+==================+=============+==========+============+=======================+===============+
| Stochastic Gradient Descent |              207.58 |          90.93 |             9.07 |        0.94 |     0.91 |       0.88 |                     0 |             0 |
+-----------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [20]:
def train_adaboost(X_train, y_train, X_test):
    start_time = time.time()

    model = AdaBoostClassifier(random_state=42)
    model.fit(X_train, y_train)

    training_time = time.time() - start_time

    y_pred = model.predict(X_test)

    return y_pred, training_time

y_pred, training_time = train_adaboost(X_train, y_train, X_test)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Ada Boost')
print(metrics_table)

+-----------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model     |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+===========+=====================+================+==================+=============+==========+============+=======================+===============+
| Ada Boost |              223.72 |          90.93 |             9.07 |        0.94 |     0.91 |       0.88 |                     0 |             0 |
+-----------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [21]:
def train_gradient_boosting(X_train, y_train, X_test):
    start_time = time.time()

    model = GradientBoostingClassifier(random_state=42)
    model.fit(X_train, y_train)

    training_time = time.time() - start_time

    y_pred = model.predict(X_test)

    return y_pred, training_time
y_pred, training_time = train_gradient_boosting(X_train, y_train, X_test)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Gradient Boosting')
print(metrics_table)

In [23]:
def train_ann(X_train, y_train, X_test, y_test, input_dim, num_classes):
    start_time = time.time()

    model = Sequential()
    model.add(Dense(64, input_dim=input_dim, activation='relu'))  
    model.add(Dense(64, activation='relu'))  
    model.add(Dense(num_classes, activation='softmax')) 

    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    model.fit(X_train, y_train, epochs=10, batch_size=32, verbose=0)

    training_time = time.time() - start_time

    y_pred_prob = model.predict(X_test)
    y_pred = np.argmax(y_pred_prob, axis=1) 
    accuracy = accuracy_score(y_test, y_pred)
    print(f'Accuracy: {accuracy * 100:.2f}%')

    return y_pred, training_time

num_classes = len(set(y_train.astype(str)))  
y_pred, training_time = train_ann(X_train, y_train_encoded, X_test, y_test_encoded, input_dim=X_train.shape[1], num_classes=num_classes)
metrics_table = calculate_metrics(y_test_encoded, y_pred, training_time, 'weighted', 'Artificial Neural Network')
print(metrics_table)

c:\Users\pc\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


9547/9547 ━━━━━━━━━━━━━━━━━━━━ 7s 701us/step
Accuracy: 90.67%


+---------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model                     |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+===========================+=====================+================+==================+=============+==========+============+=======================+===============+
| Artificial Neural Network |               198.9 |          90.67 |             9.33 |        0.93 |     0.91 |       0.88 |                     0 |             0 |
+---------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [25]:
def train_deep_ann(X_train, y_train, X_test, y_test, input_dim, num_classes):
    start_time = time.time()


    model = Sequential()
    model.add(Dense(128, input_dim=input_dim, activation='relu')) 
    model.add(Dropout(0.2)) 
    model.add(Dense(128, activation='relu')) 
    model.add(Dense(64, activation='relu')) 
    model.add(Dense(32, activation='relu'))
    model.add(Dense(num_classes, activation='softmax'))  

    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    model.fit(X_train, y_train, epochs=20, batch_size=32, verbose=0)

    training_time = time.time() - start_time

    y_pred_prob = model.predict(X_test)
    y_pred = np.argmax(y_pred_prob, axis=1)

    accuracy = accuracy_score(y_test, y_pred)
    print(f'Accuracy: {accuracy * 100:.2f}%')

    return y_pred, training_time

num_classes = len(set(y_train.astype(str)))
y_pred, training_time = train_deep_ann(X_train, y_train_encoded, X_test, y_test_encoded, input_dim=X_train.shape[1], num_classes=num_classes)
metrics_table = calculate_metrics(y_test_encoded, y_pred, training_time, 'weighted', 'Deep Artificial Neural Network')
print(metrics_table)

c:\Users\pc\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


9547/9547 ━━━━━━━━━━━━━━━━━━━━ 7s 705us/step
Accuracy: 88.02%


+--------------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model                          |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+================================+=====================+================+==================+=============+==========+============+=======================+===============+
| Deep Artificial Neural Network |              522.42 |          88.02 |            11.98 |        0.91 |     0.88 |       0.85 |                     0 |             0 |
+--------------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [35]:
def train_cnn_lstm(X_train, y_train, X_test, y_test, input_shape, num_classes):
    start_time = time.time()

    model = Sequential()

    model.add(Conv1D(filters=64, kernel_size=3, activation='relu', input_shape=input_shape))
    model.add(MaxPooling1D(pool_size=2))
    model.add(Dropout(0.3))

    model.add(Conv1D(filters=64, kernel_size=3, activation='relu'))
    model.add(MaxPooling1D(pool_size=2))
    model.add(Dropout(0.3))

    model.add(LSTM(50, activation='relu', return_sequences=False))
    model.add(Dropout(0.3))

    model.add(Dense(50, activation='relu'))
    model.add(Dense(num_classes, activation='softmax'))

    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

    model.fit(X_train, y_train, epochs=20, batch_size=64, verbose=1, validation_split=0.2, callbacks=[early_stopping])

    training_time = time.time() - start_time

    y_pred_prob = model.predict(X_test)
    y_pred = np.argmax(y_pred_prob, axis=1)

    accuracy = accuracy_score(y_test, y_pred)
    print(f'Accuracy: {accuracy * 100:.2f}%')

    return y_pred, training_time

label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

X_train_cnn_lstm = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))
X_test_cnn_lstm = X_test.reshape((X_test.shape[0], X_test.shape[1], 1))

num_classes = len(set(y_train_encoded))

y_pred, training_time = train_cnn_lstm(X_train_cnn_lstm, y_train_encoded, X_test_cnn_lstm, y_test_encoded, input_shape=(X_train.shape[1], 1), num_classes=num_classes)

metrics_table = calculate_metrics(y_test_encoded, y_pred, training_time, 'weighted', 'Convolutional Neural Network')
print(metrics_table)

Epoch 1/20


c:\Users\pc\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


674/674 ━━━━━━━━━━━━━━━━━━━━ 11s 14ms/step - accuracy: 0.3900 - loss: 1.4735 - val_accuracy: 0.7273 - val_loss: 0.4823
Epoch 2/20
674/674 ━━━━━━━━━━━━━━━━━━━━ 9s 14ms/step - accuracy: 0.7113 - loss: 0.5219 - val_accuracy: 0.7608 - val_loss: 0.4353
Epoch 3/20
674/674 ━━━━━━━━━━━━━━━━━━━━ 9s 13ms/step - accuracy: 0.7389 - loss: 0.4677 - val_accuracy: 0.7197 - val_loss: 0.4418
Epoch 4/20
674/674 ━━━━━━━━━━━━━━━━━━━━ 10s 14ms/step - accuracy: 0.7576 - loss: 0.4333 - val_accuracy: 0.7485 - val_loss: 0.3968
Epoch 5/20
674/674 ━━━━━━━━━━━━━━━━━━━━ 10s 14ms/step - accuracy: 0.7736 - loss: 0.4006 - val_accuracy: 0.7610 - val_loss: 0.4176
Epoch 6/20
674/674 ━━━━━━━━━━━━━━━━━━━━ 9s 14ms/step - accuracy: 0.7822 - loss: 0.3745 - val_accuracy: 0.7950 - val_loss: 0.3490
Epoch 7/20
674/674 ━━━━━━━━━━━━━━━━━━━━ 10s 15ms/step - accuracy: 0.7891 - loss: 0.3707 - val_accuracy: 0.7751 - val_loss: 0.3753
Epoch 8/20
674/674 ━━━━━━━━━━━━━━━━━━━━ 10s 14ms/step - accuracy: 0.7940 - loss: 0.3604 - val_accuracy: 

c:\Users\pc\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


## USING PCA FOR THE MODELS

In [36]:
from sklearn.decomposition import PCA

In [37]:
pca = PCA(n_components=12)
x_train = pca.fit_transform(X_train)
x_test = pca.transform(X_test)
tune_value = 0.20

In [31]:
def train_model(X_train, y_train, X_test):
    start_time = time.time()  

    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)

    training_time = time.time() - start_time

    y_pred = model.predict(X_test)

    return y_pred, training_time

y_pred, training_time = train_model(X_train, y_train, X_test)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Random Forest')
print(metrics_table)

+---------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model         |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+===============+=====================+================+==================+=============+==========+============+=======================+===============+
| Random Forest |              215.05 |          90.93 |             9.07 |        0.94 |     0.91 |       0.88 |                     0 |             0 |
+---------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [32]:
def train_logistic_regression(X_train, y_train, X_test):
    start_time = time.time()

    model = LogisticRegression(random_state=42, max_iter=1000)
    model.fit(X_train, y_train)

    training_time = time.time() - start_time

    y_pred = model.predict(X_test)

    return y_pred, training_time
y_pred, training_time = train_logistic_regression(X_train, y_train, X_test)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Logistic Regression')
print(metrics_table)

+---------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model               |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+=====================+=====================+================+==================+=============+==========+============+=======================+===============+
| Logistic Regression |              217.76 |          90.93 |             9.07 |        0.94 |     0.91 |       0.88 |                     0 |             0 |
+---------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [33]:
def train_svm(X_train, y_train, X_test):
    start_time = time.time()

    model = SVC(random_state=42)
    model.fit(X_train, y_train)

    training_time = time.time() - start_time

    y_pred = model.predict(X_test)

    return y_pred, training_time

y_pred, training_time = train_svm(X_train, y_train, X_test)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Support Vector Machine')
print(metrics_table)

+------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model                  |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+========================+=====================+================+==================+=============+==========+============+=======================+===============+
| Support Vector Machine |              212.98 |          90.93 |             9.07 |        0.94 |     0.91 |       0.88 |                     0 |             0 |
+------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [34]:
def train_sgd(X_train, y_train, X_test):
    start_time = time.time()

    model = SGDClassifier(random_state=42)
    model.fit(X_train, y_train)

    training_time = time.time() - start_time

    y_pred = model.predict(X_test)

    return y_pred, training_time
y_pred, training_time = train_sgd(X_train, y_train, X_test)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Stochastic Gradient Descent')
print(metrics_table)

+-----------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model                       |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+=============================+=====================+================+==================+=============+==========+============+=======================+===============+
| Stochastic Gradient Descent |              207.37 |          90.93 |             9.07 |        0.94 |     0.91 |       0.88 |                     0 |             0 |
+-----------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [35]:
def train_adaboost(X_train, y_train, X_test):
    start_time = time.time()

    model = AdaBoostClassifier(random_state=42)
    model.fit(X_train, y_train)

    training_time = time.time() - start_time

    y_pred = model.predict(X_test)

    return y_pred, training_time

y_pred, training_time = train_adaboost(X_train, y_train, X_test)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Ada Boost')
print(metrics_table)

+-----------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model     |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+===========+=====================+================+==================+=============+==========+============+=======================+===============+
| Ada Boost |              208.62 |          90.93 |             9.07 |        0.94 |     0.91 |       0.88 |                     0 |             0 |
+-----------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [36]:
def train_gradient_boosting(X_train, y_train, X_test):
    start_time = time.time()

    model = GradientBoostingClassifier(random_state=42)
    model.fit(X_train, y_train)

    training_time = time.time() - start_time

    y_pred = model.predict(X_test)

    return y_pred, training_time
y_pred, training_time = train_gradient_boosting(X_train, y_train, X_test)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Gradient Boosting')
print(metrics_table)

+-------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model             |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+===================+=====================+================+==================+=============+==========+============+=======================+===============+
| Gradient Boosting |              217.98 |          90.93 |             9.07 |        0.94 |     0.91 |       0.88 |                     0 |             0 |
+-------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [37]:
def train_ann(X_train, y_train, X_test, y_test, input_dim, num_classes):
    start_time = time.time()

    model = Sequential()
    model.add(Dense(64, input_dim=input_dim, activation='relu'))  
    model.add(Dense(64, activation='relu'))  
    model.add(Dense(num_classes, activation='softmax')) 

    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    model.fit(X_train, y_train, epochs=10, batch_size=32, verbose=0)

    training_time = time.time() - start_time

    y_pred_prob = model.predict(X_test)
    y_pred = np.argmax(y_pred_prob, axis=1) 
    accuracy = accuracy_score(y_test, y_pred)
    print(f'Accuracy: {accuracy * 100:.2f}%')

    return y_pred, training_time

num_classes = len(set(y_train.astype(str)))  
y_pred, training_time = train_ann(X_train, y_train_encoded, X_test, y_test_encoded, input_dim=X_train.shape[1], num_classes=num_classes)
metrics_table = calculate_metrics(y_test_encoded, y_pred, training_time, 'weighted', 'Artificial Neural Network')
print(metrics_table)

c:\Users\pc\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


9547/9547 ━━━━━━━━━━━━━━━━━━━━ 9s 923us/step
Accuracy: 90.62%
+---------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model                     |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+===========================+=====================+================+==================+=============+==========+============+=======================+===============+
| Artificial Neural Network |              211.83 |          90.62 |             9.38 |        0.94 |     0.91 |       0.87 |                     0 |             0 |
+---------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [49]:
def train_deep_ann(X_train, y_train, X_test, y_test, input_dim, num_classes):
    start_time = time.time()


    model = Sequential()
    model.add(Dense(128, input_dim=input_dim, activation='relu')) 
    model.add(Dropout(0.2)) 
    model.add(Dense(128, activation='relu')) 
    model.add(Dense(64, activation='relu')) 
    model.add(Dense(32, activation='relu'))
    model.add(Dense(num_classes, activation='softmax'))  

    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    model.fit(X_train, y_train, epochs=20, batch_size=32, verbose=0)

    training_time = time.time() - start_time

    y_pred_prob = model.predict(X_test)
    y_pred = np.argmax(y_pred_prob, axis=1)

    accuracy = accuracy_score(y_test, y_pred)
    print(f'Accuracy: {accuracy * 100:.2f}%')

    return y_pred, training_time

num_classes = len(set(y_train.astype(str)))
y_pred, training_time = train_deep_ann(X_train, y_train_encoded, X_test, y_test_encoded, input_dim=X_train.shape[1], num_classes=num_classes)
metrics_table = calculate_metrics(y_test_encoded, y_pred, training_time, 'weighted', 'Deep Artificial Neural Network')
print(metrics_table)

c:\Users\pc\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


9547/9547 ━━━━━━━━━━━━━━━━━━━━ 7s 757us/step
Accuracy: 87.31%
+--------------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model                          |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+================================+=====================+================+==================+=============+==========+============+=======================+===============+
| Deep Artificial Neural Network |              543.59 |          87.31 |            12.69 |        0.92 |     0.87 |       0.84 |                     0 |             0 |
+--------------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [38]:
def train_cnn_lstm(X_train, y_train, X_test, y_test, input_shape, num_classes):
    start_time = time.time()

    model = Sequential()

    model.add(Conv1D(filters=64, kernel_size=3, activation='relu', input_shape=input_shape))
    model.add(MaxPooling1D(pool_size=2))
    model.add(Dropout(0.3))

    model.add(Conv1D(filters=64, kernel_size=3, activation='relu'))
    model.add(MaxPooling1D(pool_size=2))
    model.add(Dropout(0.3))

    model.add(LSTM(50, activation='relu', return_sequences=False))
    model.add(Dropout(0.3))

    model.add(Dense(50, activation='relu'))
    model.add(Dense(num_classes, activation='softmax'))

    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

    model.fit(X_train, y_train, epochs=20, batch_size=64, verbose=1, validation_split=0.2, callbacks=[early_stopping])

    training_time = time.time() - start_time

    y_pred_prob = model.predict(X_test)
    y_pred = np.argmax(y_pred_prob, axis=1)

    accuracy = accuracy_score(y_test, y_pred)
    print(f'Accuracy: {accuracy * 100:.2f}%')

    return y_pred, training_time

label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

X_train_cnn_lstm = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))
X_test_cnn_lstm = X_test.reshape((X_test.shape[0], X_test.shape[1], 1))

num_classes = len(set(y_train_encoded))

y_pred, training_time = train_cnn_lstm(X_train_cnn_lstm, y_train_encoded, X_test_cnn_lstm, y_test_encoded, input_shape=(X_train.shape[1], 1), num_classes=num_classes)

metrics_table = calculate_metrics(y_test_encoded, y_pred, training_time, 'weighted', 'Convolutional Neural Network')
print(metrics_table)

Epoch 1/20


c:\Users\pc\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


674/674 ━━━━━━━━━━━━━━━━━━━━ 11s 13ms/step - accuracy: 0.3574 - loss: 1.6141 - val_accuracy: 0.6718 - val_loss: 0.6038
Epoch 2/20
674/674 ━━━━━━━━━━━━━━━━━━━━ 9s 13ms/step - accuracy: 0.6974 - loss: 0.5646 - val_accuracy: 0.7562 - val_loss: 0.4147
Epoch 3/20
674/674 ━━━━━━━━━━━━━━━━━━━━ 9s 13ms/step - accuracy: 0.7431 - loss: 0.4534 - val_accuracy: 0.7447 - val_loss: 0.4148
Epoch 4/20
674/674 ━━━━━━━━━━━━━━━━━━━━ 9s 13ms/step - accuracy: 0.7499 - loss: 0.4589 - val_accuracy: 0.7329 - val_loss: 0.4218
Epoch 5/20
674/674 ━━━━━━━━━━━━━━━━━━━━ 9s 14ms/step - accuracy: 0.7689 - loss: 0.4013 - val_accuracy: 0.7912 - val_loss: 0.3517
Epoch 6/20
674/674 ━━━━━━━━━━━━━━━━━━━━ 10s 14ms/step - accuracy: 0.7786 - loss: 0.3803 - val_accuracy: 0.8133 - val_loss: 0.3291
Epoch 7/20
674/674 ━━━━━━━━━━━━━━━━━━━━ 9s 13ms/step - accuracy: 0.7832 - loss: 0.3867 - val_accuracy: 0.7841 - val_loss: 0.3672
Epoch 8/20
674/674 ━━━━━━━━━━━━━━━━━━━━ 10s 14ms/step - accuracy: 0.7926 - loss: 0.3688 - val_accuracy: 0.

c:\Users\pc\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


## USING SMOTE FOR THE MODELS 

In [39]:
from imblearn.over_sampling import SMOTE

In [40]:
smote = SMOTE(random_state=42)
x_train, y_train = smote.fit_resample(X_train, y_train)

In [41]:
def train_model(X_train, y_train, X_test):
    start_time = time.time()  

    model = RandomForestClassifier(n_estimators=100, random_state=42)
    model.fit(X_train, y_train)

    training_time = time.time() - start_time

    y_pred = model.predict(X_test)

    return y_pred, training_time

y_pred, training_time = train_model(X_train, y_train, X_test)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Random Forest')
print(metrics_table)

+---------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model         |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+===============+=====================+================+==================+=============+==========+============+=======================+===============+
| Random Forest |              206.46 |          90.93 |             9.07 |        0.94 |     0.91 |       0.88 |                     0 |             0 |
+---------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [42]:
def train_logistic_regression(X_train, y_train, X_test):
    start_time = time.time()

    model = LogisticRegression(random_state=42, max_iter=1000)
    model.fit(X_train, y_train)

    training_time = time.time() - start_time

    y_pred = model.predict(X_test)

    return y_pred, training_time
y_pred, training_time = train_model(X_train, y_train, X_test)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Logistic Regression')
print(metrics_table)

+---------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model               |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+=====================+=====================+================+==================+=============+==========+============+=======================+===============+
| Logistic Regression |              206.24 |          90.93 |             9.07 |        0.94 |     0.91 |       0.88 |                     0 |             0 |
+---------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [43]:
def train_svm(X_train, y_train, X_test):
    start_time = time.time()

    model = SVC(random_state=42)
    model.fit(X_train, y_train)

    training_time = time.time() - start_time

    y_pred = model.predict(X_test)

    return y_pred, training_time

y_pred, training_time = train_model(X_train, y_train, X_test)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Support Vector Machine')
print(metrics_table)

+------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model                  |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+========================+=====================+================+==================+=============+==========+============+=======================+===============+
| Support Vector Machine |               206.8 |          90.93 |             9.07 |        0.94 |     0.91 |       0.88 |                     0 |             0 |
+------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [44]:
def train_sgd(X_train, y_train, X_test):
    start_time = time.time()

    model = SGDClassifier(random_state=42)
    model.fit(X_train, y_train)

    training_time = time.time() - start_time

    y_pred = model.predict(X_test)

    return y_pred, training_time
y_pred, training_time = train_model(X_train, y_train, X_test)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Stochastic Gradient Descent')
print(metrics_table)

+-----------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model                       |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+=============================+=====================+================+==================+=============+==========+============+=======================+===============+
| Stochastic Gradient Descent |              206.37 |          90.93 |             9.07 |        0.94 |     0.91 |       0.88 |                     0 |             0 |
+-----------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [45]:
def train_adaboost(X_train, y_train, X_test):
    start_time = time.time()

    model = AdaBoostClassifier(random_state=42)
    model.fit(X_train, y_train)

    training_time = time.time() - start_time

    y_pred = model.predict(X_test)

    return y_pred, training_time

y_pred, training_time = train_model(X_train, y_train, X_test)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Ada Boost')
print(metrics_table)

+-----------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model     |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+===========+=====================+================+==================+=============+==========+============+=======================+===============+
| Ada Boost |              206.73 |          90.93 |             9.07 |        0.94 |     0.91 |       0.88 |                     0 |             0 |
+-----------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [46]:
def train_gradient_boosting(X_train, y_train, X_test):
    start_time = time.time()

    model = GradientBoostingClassifier(random_state=42)
    model.fit(X_train, y_train)

    training_time = time.time() - start_time

    y_pred = model.predict(X_test)

    return y_pred, training_time
y_pred, training_time = train_model(X_train, y_train, X_test)
metrics_table = calculate_metrics(y_test, y_pred, training_time, 'weighted', 'Gradient Boosting')
print(metrics_table)

+-------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model             |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+===================+=====================+================+==================+=============+==========+============+=======================+===============+
| Gradient Boosting |              206.19 |          90.93 |             9.07 |        0.94 |     0.91 |       0.88 |                     0 |             0 |
+-------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [47]:
def train_ann(X_train, y_train, X_test, y_test, input_dim, num_classes):
    start_time = time.time()

    model = Sequential()
    model.add(Dense(64, input_dim=input_dim, activation='relu'))  
    model.add(Dense(64, activation='relu'))  
    model.add(Dense(num_classes, activation='softmax')) 

    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    model.fit(X_train, y_train, epochs=10, batch_size=32, verbose=0)

    training_time = time.time() - start_time

    y_pred_prob = model.predict(X_test)
    y_pred = np.argmax(y_pred_prob, axis=1) 
    accuracy = accuracy_score(y_test, y_pred)
    print(f'Accuracy: {accuracy * 100:.2f}%')

    return y_pred, training_time

num_classes = len(set(y_train.astype(str)))  
y_pred, training_time = train_ann(X_train, y_train_encoded, X_test, y_test_encoded, input_dim=X_train.shape[1], num_classes=num_classes)
metrics_table = calculate_metrics(y_test_encoded, y_pred, training_time, 'weighted', 'Artificial Neural Network')
print(metrics_table)

c:\Users\pc\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


9547/9547 ━━━━━━━━━━━━━━━━━━━━ 6s 668us/step
Accuracy: 90.71%
+---------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model                     |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+===========================+=====================+================+==================+=============+==========+============+=======================+===============+
| Artificial Neural Network |              203.31 |          90.71 |             9.29 |        0.92 |     0.91 |       0.88 |                     0 |             0 |
+---------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [48]:
def train_deep_ann(X_train, y_train, X_test, y_test, input_dim, num_classes):
    start_time = time.time()


    model = Sequential()
    model.add(Dense(128, input_dim=input_dim, activation='relu')) 
    model.add(Dropout(0.2)) 
    model.add(Dense(128, activation='relu')) 
    model.add(Dense(64, activation='relu')) 
    model.add(Dense(32, activation='relu'))
    model.add(Dense(num_classes, activation='softmax'))  

    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    model.fit(X_train, y_train, epochs=20, batch_size=32, verbose=0)

    training_time = time.time() - start_time

    y_pred_prob = model.predict(X_test)
    y_pred = np.argmax(y_pred_prob, axis=1)

    accuracy = accuracy_score(y_test, y_pred)
    print(f'Accuracy: {accuracy * 100:.2f}%')

    return y_pred, training_time

num_classes = len(set(y_train.astype(str)))
y_pred, training_time = train_deep_ann(X_train, y_train_encoded, X_test, y_test_encoded, input_dim=X_train.shape[1], num_classes=num_classes)
metrics_table = calculate_metrics(y_test_encoded, y_pred, training_time, 'weighted', 'Deep Artificial Neural Network')
print(metrics_table)

c:\Users\pc\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


9547/9547 ━━━━━━━━━━━━━━━━━━━━ 7s 762us/step
Accuracy: 82.88%
+--------------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+
| Model                          |   Training Time (s) |   Accuracy (%) |   Error Rate (%) |   Precision |   Recall |   F1-Score |   False Positive Rate |   Specificity |
+================================+=====================+================+==================+=============+==========+============+=======================+===============+
| Deep Artificial Neural Network |              553.13 |          82.88 |            17.12 |        0.86 |     0.83 |       0.79 |                     0 |             0 |
+--------------------------------+---------------------+----------------+------------------+-------------+----------+------------+-----------------------+---------------+


In [41]:
def train_cnn_lstm(X_train, y_train, X_test, y_test, input_shape, num_classes):
    start_time = time.time()

    model = Sequential()

    model.add(Conv1D(filters=64, kernel_size=3, activation='relu', input_shape=input_shape))
    model.add(MaxPooling1D(pool_size=2))
    model.add(Dropout(0.3))

    model.add(Conv1D(filters=64, kernel_size=3, activation='relu'))
    model.add(MaxPooling1D(pool_size=2))
    model.add(Dropout(0.3))

    model.add(LSTM(50, activation='relu', return_sequences=False))
    model.add(Dropout(0.3))

    model.add(Dense(50, activation='relu'))
    model.add(Dense(num_classes, activation='softmax'))

    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)

    model.fit(X_train, y_train, epochs=20, batch_size=64, verbose=1, validation_split=0.2, callbacks=[early_stopping])

    training_time = time.time() - start_time

    y_pred_prob = model.predict(X_test)
    y_pred = np.argmax(y_pred_prob, axis=1)

    accuracy = accuracy_score(y_test, y_pred)
    print(f'Accuracy: {accuracy * 100:.2f}%')

    return y_pred, training_time

label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

X_train_cnn_lstm = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))
X_test_cnn_lstm = X_test.reshape((X_test.shape[0], X_test.shape[1], 1))

num_classes = len(set(y_train_encoded))

y_pred, training_time = train_cnn_lstm(X_train_cnn_lstm, y_train_encoded, X_test_cnn_lstm, y_test_encoded, input_shape=(X_train.shape[1], 1), num_classes=num_classes)

metrics_table = calculate_metrics(y_test_encoded, y_pred, training_time, 'weighted', 'Convolutional Neural Network')
print(metrics_table)

Epoch 1/20


c:\Users\pc\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


674/674 ━━━━━━━━━━━━━━━━━━━━ 12s 14ms/step - accuracy: 0.3630 - loss: 1.5699 - val_accuracy: 0.7243 - val_loss: 0.5145
Epoch 2/20
674/674 ━━━━━━━━━━━━━━━━━━━━ 9s 14ms/step - accuracy: 0.7020 - loss: 0.5702 - val_accuracy: 0.6785 - val_loss: 0.5164
Epoch 3/20
674/674 ━━━━━━━━━━━━━━━━━━━━ 9s 13ms/step - accuracy: 0.7316 - loss: 0.4847 - val_accuracy: 0.7437 - val_loss: 0.4344
Epoch 4/20
674/674 ━━━━━━━━━━━━━━━━━━━━ 9s 13ms/step - accuracy: 0.7560 - loss: 0.4278 - val_accuracy: 0.7154 - val_loss: 0.4917
Epoch 5/20
674/674 ━━━━━━━━━━━━━━━━━━━━ 9s 14ms/step - accuracy: 0.7746 - loss: 0.3919 - val_accuracy: 0.8014 - val_loss: 0.3408
Epoch 6/20
674/674 ━━━━━━━━━━━━━━━━━━━━ 9s 13ms/step - accuracy: 0.7890 - loss: 0.3728 - val_accuracy: 0.7018 - val_loss: 0.6366
Epoch 7/20
674/674 ━━━━━━━━━━━━━━━━━━━━ 9s 14ms/step - accuracy: 0.7851 - loss: 0.4015 - val_accuracy: 0.7755 - val_loss: 0.3661
Epoch 8/20
674/674 ━━━━━━━━━━━━━━━━━━━━ 9s 14ms/step - accuracy: 0.7979 - loss: 0.3575 - val_accuracy: 0.80